<a href="https://colab.research.google.com/github/kartik2006-del/efficient-finetuning-lora/blob/main/lora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##  Target-module comparison



In [1]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!pip uninstall torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Would remove:
    /usr/local/lib/python3.13/dist-packages/torchao-0.10.0.dist-info/*
    /usr/local/lib/python3.13/dist-packages/torchao/*
Proceed (Y/n)? Y
  Successfully uninstalled torchao-0.10.0


In [11]:
from datasets import load_dataset, Dataset

dataset = load_dataset(
    "yahma/alpaca-cleaned",
    split="train"
)

df = dataset.to_pandas()
df = df.drop_duplicates()

dataset = Dataset.from_pandas(
    df,
    preserve_index=False
)

train_test = dataset.train_test_split(
    test_size=0.20,
    seed=42
)

train_data = train_test["train"]
temp_data = train_test["test"]

validation_test = temp_data.train_test_split(
    test_size=0.50,
    seed=42
)

validation_data = validation_test["train"]
test_data = validation_test["test"]

print("Train:", len(train_data))
print("Validation:", len(validation_data))
print("Test:", len(test_data))

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

Train: 41404
Validation: 5176
Test: 5176


In [12]:
from transformers import AutoTokenizer, DataCollatorForLanguageModeling

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token


def format_chat(example):
    if example["input"].strip():
        user_content = (
            example["instruction"]
            + "\n\nInput:\n"
            + example["input"]
        )
    else:
        user_content = example["instruction"]

    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": example["output"]}
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False
        )
    }


def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512,
        padding="max_length"
    )


validation_test_data = validation_data.select(range(100))

validation_test_data = validation_test_data.map(
    format_chat
)

validation_test_data = validation_test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=validation_test_data.column_names
)


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Setup restored successfully.")

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Setup restored successfully.


In [14]:
full_ft_test_data = train_data.select(range(100))

full_ft_test_data = full_ft_test_data.map(
    format_chat
)

full_ft_test_data = full_ft_test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=full_ft_test_data.column_names
)

print("Training examples:", len(full_ft_test_data))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Training examples: 100


In [15]:
import gc
import time
import torch
import pandas as pd

from transformers import (
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import LoraConfig, get_peft_model

target_configs = {
    "Q + V": ["q_proj", "v_proj"],
    "Q + K + V": ["q_proj", "k_proj", "v_proj"],
    "Q + K + V + O": ["q_proj", "k_proj", "v_proj", "o_proj"]
}


validation_test_data = validation_data.select(range(100))

validation_test_data = validation_test_data.map(
    format_chat
)

validation_test_data = validation_test_data.map(
    tokenize_function,
    batched=True,
    remove_columns=validation_test_data.column_names
)


data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


target_training_results = []

for name, targets in target_configs.items():

    print("\n" + "=" * 50)
    print(f"TRAINING: {name}")
    print("=" * 50)

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32,
        device_map="auto"
    )

    config = LoraConfig(
        r=16,
        lora_alpha=32,
        lora_dropout=0.05,
        target_modules=targets,
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, config)

    trainable = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    training_args = TrainingArguments(
        output_dir=f"./outputs/target_module_{name.replace(' + ', '_')}",
        max_steps=20,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=5,
        save_strategy="no",
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=full_ft_test_data,
        eval_dataset=validation_test_data,
        data_collator=data_collator
    )

    start_time = time.time()

    train_output = trainer.train()

    training_time = time.time() - start_time

    eval_output = trainer.evaluate()

    eval_loss = eval_output["eval_loss"]

    target_training_results.append({
        "target_modules": name,
        "trainable_params": trainable,
        "training_time_sec": training_time,
        "train_loss": train_output.training_loss,
        "eval_loss": eval_loss
    })

    print(f"\nTrainable params: {trainable:,}")
    print(f"Training time: {training_time:.2f} sec")
    print(f"Train loss: {train_output.training_loss:.4f}")
    print(f"Validation loss: {eval_loss:.4f}")

    # Free memory
    del trainer
    del model
    gc.collect()
    torch.cuda.empty_cache()

target_results_df = pd.DataFrame(target_training_results)

target_results_df

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]


TRAINING: Q + V


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
5,2.334442
10,1.964664
15,1.698326
20,1.913639


...

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Step,Training Loss
5,2.253869
10,1.856994
15,1.562878
20,1.659730


Training Loss,Validation Loss,Step
1.659730,1.370771,20



Trainable params: 2,162,688
Training time: 4.81 sec
Train loss: 1.8334
Validation loss: 1.3708


,target_modules,trainable_params,training_time_sec,train_loss,eval_loss
0,Q + V,1081344,4.729441,1.977768,1.549908
1,Q + K + V,1474560,4.118824,1.952317,1.520788
2,Q + K + V + O,2162688,4.807529,1.833368,1.370771
